In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
    #for filename in filenames:
        #print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import torch
from torch import nn
import torch.nn.functional as F

In [3]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
print(f'device={device}')

device=cuda


In [ ]:
#8head,0extract
#(height//patch_size)*(width//patch_size)=256
#input_dim= (height//patch_size)*(width//patch_size)+1=257#一张图像分割成多少个块进入extractAttention模块，即该模块的输入维度
#output_dim=32
class extractAttention(nn.Module):
    def __init__(self,  input_dim, output_dim):
        super().__init__()
        self.query = nn.Linear(input_dim, output_dim)
        self.key = nn.Linear(input_dim, output_dim)
        self.value = nn.Linear(input_dim, output_dim)
        

    def forward(self, x):
        
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        
        kweight=self.key.weight
        
        attn_weights1 = F.softmax(q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5), dim=-1)
        Vmix0 = attn_weights1 @ v
        
        return Vmix0
     


class extractMultihead(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.Head0=extractAttention( input_dim, output_dim )
        self.Head1=extractAttention( input_dim, output_dim )
        self.Head2=extractAttention( input_dim, output_dim )
        self.Head3=extractAttention( input_dim, output_dim )
        self.Head4=extractAttention( input_dim, output_dim )
        self.Head5=extractAttention( input_dim, output_dim )
        self.Head6=extractAttention( input_dim, output_dim )
        self.Head7=extractAttention( input_dim, output_dim )
        
    def forward(self,x):
        temp0=self.Head0(x)
        temp1=self.Head1(x)
        temp2=self.Head2(x)
        temp3=self.Head3(x)
        temp4=self.Head4(x)
        temp5=self.Head5(x)
        temp6=self.Head6(x)
        temp7=self.Head7(x)
        
        temp=(temp0+temp1+temp2+temp3+temp4+temp5+temp6+temp7)/8
        
        return temp


In [ ]:
#8head,1extract
#(height//patch_size)*(width//patch_size)=256
#input_dim= (height//patch_size)*(width//patch_size)+1=257#一张图像分割成多少个块进入extractAttention模块，即该模块的输入维度
#output_dim=32
class extractAttention(nn.Module):
    def __init__(self,  input_dim, output_dim):
        super().__init__()
        self.query = nn.Linear(input_dim, output_dim)
        self.key = nn.Linear(input_dim, output_dim)
        self.value = nn.Linear(input_dim, output_dim)
        

    def forward(self, x):
        
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        
        kweight=self.key.weight
        
        attn_weights0 = F.softmax(q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5), dim=-1)
        Vmix0 = attn_weights0 @ v
        
        k1=self.extract(k,kweight)
        attn_weights1 = F.softmax(q @ k1.transpose(-2, -1) / (k1.shape[-1] ** 0.5), dim=-1)
        Vmix1 = attn_weights1 @ v
        

        Vmix=(Vmix0+Vmix1)/2
        
        
        return Vmix
    def extract(self,k,kweight):
        with torch.no_grad():
            return torch.matmul(torch.matmul(k,kweight),torch.transpose(kweight,0,1))  
            #很不幸，这玩意是行向量。
            #WW_tk=W(k_tW)_t=[(k_tW)W_t]_t


class extractMultihead(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.Head0=extractAttention( input_dim, output_dim )
        self.Head1=extractAttention( input_dim, output_dim )
        self.Head2=extractAttention( input_dim, output_dim )
        self.Head3=extractAttention( input_dim, output_dim )
        self.Head4=extractAttention( input_dim, output_dim )
        self.Head5=extractAttention( input_dim, output_dim )
        self.Head6=extractAttention( input_dim, output_dim )
        self.Head7=extractAttention( input_dim, output_dim )

        
    def forward(self,x):
        temp0=self.Head0(x)
        temp1=self.Head1(x)
        temp2=self.Head2(x)
        temp3=self.Head3(x)
        temp4=self.Head4(x)
        temp5=self.Head5(x)
        temp6=self.Head6(x)
        temp7=self.Head7(x)

       #temp=(temp0+temp1+temp2+temp3)/4
        temp=(temp0+temp1+temp2+temp3+temp4+temp5+temp6+temp7)/8
        
        return temp


In [4]:
#8head,2extract
#(height//patch_size)*(width//patch_size)=256
#input_dim= (height//patch_size)*(width//patch_size)+1=257#一张图像分割成多少个块进入extractAttention模块，即该模块的输入维度
#output_dim=32
class extractAttention(nn.Module):
    def __init__(self,  input_dim, output_dim):
        super().__init__()
        self.query = nn.Linear(input_dim, output_dim)
        self.key = nn.Linear(input_dim, output_dim)
        self.value = nn.Linear(input_dim, output_dim)
        

    def forward(self, x):
        
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        
        kweight=self.key.weight
        
        attn_weights0 = F.softmax(q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5), dim=-1)
        Vmix0 = attn_weights0 @ v
        
        k1=self.extract(k,kweight)
        attn_weights1 = F.softmax(q @ k1.transpose(-2, -1) / (k1.shape[-1] ** 0.5), dim=-1)
        Vmix1 = attn_weights1 @ v
        
        k2=self.extract(k1,kweight)
        attn_weights2 = F.softmax(q @ k2.transpose(-2, -1) / (k2.shape[-1] ** 0.5), dim=-1)
        Vmix2 = attn_weights2 @ v
        
        
        Vmix=(Vmix0+Vmix1+Vmix2)/3
        
        
        return Vmix
    def extract(self,k,kweight):
        with torch.no_grad():
            return torch.matmul(torch.matmul(k,kweight),torch.transpose(kweight,0,1))  
            #很不幸，这玩意是行向量。
            #WW_tk=W(k_tW)_t=[(k_tW)W_t]_t


class extractMultihead(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.Head0=extractAttention( input_dim, output_dim )
        self.Head1=extractAttention( input_dim, output_dim )
        self.Head2=extractAttention( input_dim, output_dim )
        self.Head3=extractAttention( input_dim, output_dim )
        self.Head4=extractAttention( input_dim, output_dim )
        self.Head5=extractAttention( input_dim, output_dim )
        self.Head6=extractAttention( input_dim, output_dim )
        self.Head7=extractAttention( input_dim, output_dim )

        
    def forward(self,x):
        temp0=self.Head0(x)
        temp1=self.Head1(x)
        temp2=self.Head2(x)
        temp3=self.Head3(x)
        temp4=self.Head4(x)
        temp5=self.Head5(x)
        temp6=self.Head6(x)
        temp7=self.Head7(x)

       
        temp=(temp0+temp1+temp2+temp3+temp4+temp5+temp6+temp7)/8
        
        return temp


In [ ]:
#8head,3extract
#(height//patch_size)*(width//patch_size)=256
#input_dim= (height//patch_size)*(width//patch_size)+1=257#一张图像分割成多少个块进入extractAttention模块，即该模块的输入维度
#output_dim=32
class extractAttention(nn.Module):
    def __init__(self,  input_dim, output_dim):
        super().__init__()
        self.query = nn.Linear(input_dim, output_dim)
        self.key = nn.Linear(input_dim, output_dim)
        self.value = nn.Linear(input_dim, output_dim)
        

    def forward(self, x):
        
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        
        kweight=self.key.weight
        
        attn_weights0 = F.softmax(q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5), dim=-1)
        Vmix0 = attn_weights0 @ v
        
        k1=self.extract(k,kweight)
        attn_weights1 = F.softmax(q @ k1.transpose(-2, -1) / (k1.shape[-1] ** 0.5), dim=-1)
        Vmix1 = attn_weights1 @ v
        
        k2=self.extract(k1,kweight)
        attn_weights2 = F.softmax(q @ k2.transpose(-2, -1) / (k2.shape[-1] ** 0.5), dim=-1)
        Vmix2 = attn_weights2 @ v
        
        k3=self.extract(k2,kweight)
        attn_weights3 = F.softmax(q @ k3.transpose(-2, -1) / (k3.shape[-1] ** 0.5), dim=-1)
        Vmix3 = attn_weights3 @ v
        
        Vmix=(Vmix0+Vmix1+Vmix2+Vmix3)/4
        
        
        return Vmix
    def extract(self,k,kweight):
        with torch.no_grad():
            return torch.matmul(torch.matmul(k,kweight),torch.transpose(kweight,0,1))  
            #很不幸，这玩意是行向量。
            #WW_tk=W(k_tW)_t=[(k_tW)W_t]_t


class extractMultihead(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.Head0=extractAttention( input_dim, output_dim )
        self.Head1=extractAttention( input_dim, output_dim )
        self.Head2=extractAttention( input_dim, output_dim )
        self.Head3=extractAttention( input_dim, output_dim )
        self.Head4=extractAttention( input_dim, output_dim )
        self.Head5=extractAttention( input_dim, output_dim )
        self.Head6=extractAttention( input_dim, output_dim )
        self.Head7=extractAttention( input_dim, output_dim )

        
    def forward(self,x):
        temp0=self.Head0(x)
        temp1=self.Head1(x)
        temp2=self.Head2(x)
        temp3=self.Head3(x)
        temp4=self.Head4(x)
        temp5=self.Head5(x)
        temp6=self.Head6(x)
        temp7=self.Head7(x)

       
        temp=(temp0+temp1+temp2+temp3+temp4+temp5+temp6+temp7)/8
        
        return temp


In [ ]:
#8head,4extract
#(height//patch_size)*(width//patch_size)=256
#input_dim= (height//patch_size)*(width//patch_size)+1=257#一张图像分割成多少个块进入extractAttention模块，即该模块的输入维度
#output_dim=32
class extractAttention(nn.Module):
    def __init__(self,  input_dim, output_dim):
        super().__init__()
        self.query = nn.Linear(input_dim, output_dim)
        self.key = nn.Linear(input_dim, output_dim)
        self.value = nn.Linear(input_dim, output_dim)
        

    def forward(self, x):
        
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)
        
        kweight=self.key.weight
        
        attn_weights0 = F.softmax(q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5), dim=-1)
        Vmix0 = attn_weights0 @ v
        
        k1=self.extract(k,kweight)
        attn_weights1 = F.softmax(q @ k1.transpose(-2, -1) / (k1.shape[-1] ** 0.5), dim=-1)
        Vmix1 = attn_weights1 @ v
        
        k2=self.extract(k1,kweight)
        attn_weights2 = F.softmax(q @ k2.transpose(-2, -1) / (k2.shape[-1] ** 0.5), dim=-1)
        Vmix2 = attn_weights2 @ v
        
        k3=self.extract(k2,kweight)
        attn_weights3 = F.softmax(q @ k3.transpose(-2, -1) / (k3.shape[-1] ** 0.5), dim=-1)
        Vmix3 = attn_weights3 @ v
        
        k4=self.extract(k3,kweight)
        attn_weights4 = F.softmax(q @ k4.transpose(-2, -1) / (k4.shape[-1] ** 0.5), dim=-1)
        Vmix4 = attn_weights4 @ v
        
        Vmix=(Vmix0+Vmix1+Vmix2+Vmix3+Vmix3)/5
        
        
        return Vmix
    def extract(self,k,kweight):
        with torch.no_grad():
            return torch.matmul(torch.matmul(k,kweight),torch.transpose(kweight,0,1))  
            #很不幸，这玩意是行向量。
            #WW_tk=W(k_tW)_t=[(k_tW)W_t]_t


class extractMultihead(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.Head0=extractAttention( input_dim, output_dim )
        self.Head1=extractAttention( input_dim, output_dim )
        self.Head2=extractAttention( input_dim, output_dim )
        self.Head3=extractAttention( input_dim, output_dim )
        self.Head4=extractAttention( input_dim, output_dim )
        self.Head5=extractAttention( input_dim, output_dim )
        self.Head6=extractAttention( input_dim, output_dim )
        self.Head7=extractAttention( input_dim, output_dim )

        
    def forward(self,x):
        temp0=self.Head0(x)
        temp1=self.Head1(x)
        temp2=self.Head2(x)
        temp3=self.Head3(x)
        temp4=self.Head4(x)
        temp5=self.Head5(x)
        temp6=self.Head6(x)
        temp7=self.Head7(x)

       
        temp=(temp0+temp1+temp2+temp3+temp4+temp5+temp6+temp7)/8
        
        return temp


In [5]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=8, out_channels=32):
        super().__init__()
        self.patch_size = patch_size
        self.projection = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=patch_size, stride=patch_size),
            nn.Flatten(2),
            
        )

    def forward(self, x):
        
        x = self.projection(x)
        
        x=torch.transpose(x,1, 2)
        return x

In [6]:
class resBlock(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.input_dim=input_dim
        self.output_dim=output_dim
        self.resfc=nn.Linear(input_dim,output_dim)

    def forward(self,x):
        out=torch.relu(x)
        out_temp=out
        out=torch.relu(self.resfc(out)+out_temp)
        return out

In [7]:
class EMH(nn.Module):
    def __init__(self, in_channels, patch_size, out_channels, img_size,input_dim, output_dim,num_classes):
        super().__init__()
        self.patch_embedding = PatchEmbedding(in_channels, patch_size, out_channels)
        
        self.positions = nn.Parameter(torch.randn((img_size[0] // patch_size) *(img_size[1] // patch_size) , out_channels))
        self.transformer = extractMultihead(input_dim, output_dim)
        self.res1=resBlock(output_dim*out_channels, output_dim*out_channels)
        self.res2=resBlock(output_dim*out_channels//2, output_dim*out_channels//2)
        self.res3=resBlock(output_dim*out_channels*2, output_dim*out_channels*2)
        self.res4=resBlock(num_classes, num_classes)
        self.net1 = nn.Sequential(
            
            nn.Flatten(1),
            nn.LayerNorm(output_dim*out_channels),
            nn.Linear(output_dim*out_channels, 2*output_dim*out_channels),
            nn.GELU(),
            nn.Linear(2*output_dim*out_channels, 2*output_dim*out_channels),
            self.res3,
            self.res3,
            nn.GELU(),
            nn.Linear(2*output_dim*out_channels, output_dim*out_channels),
            nn.GELU(),
            nn.Linear(output_dim*out_channels, num_classes),
            
            
            
        )
        self.mlp_head = nn.Sequential(
            
            nn.GELU(),
            nn.Linear(num_classes, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, _, _, _ = x.shape        
        x = self.patch_embedding(x)        
        x += self.positions        
        x=torch.transpose(x,1, 2)
        x = self.transformer(x)      
        x = self.net1(x)
        x = self.mlp_head(x)        
        return x



In [8]:
height=width=256 
img_size=(height,width)#输入图像尺寸

batch_size=32 #数据集分组，组内元素数量

in_channels,out_channels= 3,64 #特征提取卷积层的输入张量层数和输出张量层数

patch_size = 16 #图像切割卷积层分块大小

input_dim= (height//patch_size)*(width//patch_size)#一张图像分割成多少个块进入extractAttention模块，即该模块的输入维度
output_dim=32 #extractAttention模块的输出维度


num_classes= 10

epoch=100
i=0

In [9]:
from torchvision import datasets, transforms

# 定义数据预处理步骤
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [10]:
# 加载数据集
dataset = datasets.ImageFolder('/kaggle/input/tomato-disease', transform=transform)

# 创建数据加载器
dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False)

In [11]:
from torch.utils.data import random_split

# 定义训练集和测试集的大小
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

# 使用random_split进行数据集划分
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# 创建训练集和测试集的数据加载器
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=True)

In [12]:
model=EMH(in_channels, patch_size, out_channels, img_size,input_dim, output_dim,num_classes).to(device)

criterion = nn.CrossEntropyLoss()

import torch.optim as optim
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
print(f'8HE2v6-7\n')
model.eval()  # 切换到评估模式
with torch.no_grad():  # 不需要计算梯度
    correct = 0
    total = 0
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy of the model on the validation data: {format(100 * correct / total)}%\n')

            
            
model.train()  # 切换回训练模式
for t in range(epoch):
    i = i + 1
    loss_total = 0
    b = 0
    for inputs, labels in train_dataloader:
        # 将输入和标签转移到GPU上
        inputs, labels = inputs.to(device), labels.to(device)

        # 前向传播
        outputs = model(inputs)

        # 计算损失
        loss = criterion(outputs, labels).to(device)
        b = b + 1
        loss_total = loss_total + loss

        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    loss_avg = loss_total / b
    print(f'{loss_avg}')
    
    if i % 10 == 0 or loss_avg < 0.0001:
        model.eval()  # 切换到评估模式
        with torch.no_grad():  # 不需要计算梯度
            correct = 0
            total = 0
            class_correct = [0 for _ in range(num_classes)]
            class_total = [0 for _ in range(num_classes)]

            for inputs, labels in test_dataloader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                for label, prediction in zip(labels, predicted):
                    if label == prediction:
                        class_correct[label] += 1
                    class_total[label] += 1

            overall_accuracy = 100 * correct / total
            print(f'epoch[{i}] Accuracy of the model on the validation data: {overall_accuracy}%')

            for cls in range(num_classes):
                if class_total[cls] != 0:
                    class_accuracy = 100 * class_correct[cls] / class_total[cls]
                    print(f'Accuracy of class {cls}: {class_accuracy}%')
                else:
                    print(f'Accuracy of class {cls}: N/A (no samples)')

        model.train()  # 切换回训练模式



8HE2v6-7

Accuracy of the model on the validation data: 89.62004405286343%

0.01801193505525589
0.0028490819968283176
0.00021148005907889456
0.040921006351709366
0.023796604946255684
0.0066338470205664635
0.011383544653654099
0.046706125140190125
0.03436471149325371
0.009621774777770042
epoch[110] Accuracy of the model on the validation data: 90.00550660792952%
Accuracy of class 0: 93.77880184331798%
Accuracy of class 1: 62.37623762376238%
Accuracy of class 2: 98.0%
Accuracy of class 3: 89.44591029023746%
Accuracy of class 4: 96.32014719411224%
Accuracy of class 5: 85.24590163934427%
Accuracy of class 6: 72.22222222222223%
Accuracy of class 7: 87.5%
Accuracy of class 8: 87.67908309455588%
Accuracy of class 9: 85.46099290780141%
0.003956126980483532
0.00017714532441459596
0.00013830016541760415
